# `biophys_interop` — demo

A runnable walkthrough of the four-contract pipeline on **synthetic** example records:

`standardize` → `qc` → `featurize` → `Adapter`

Everything here uses the toy `examples/*.json` shipped with the package — no real or private data.

```bash
pip install -e .   # run once from the repo root
```

In [1]:
import json, numpy as np
from biophys_interop import (
    standardize, qc, featurize, Adapter, validate,
    MODALITIES, SCHEMA_VERSION, FEATURE_DIM,
)
from biophys_interop.qc import RULE_COUNT

print("schema version :", SCHEMA_VERSION)
print("modalities     :", len(MODALITIES), "->", ", ".join(MODALITIES[:9]), "...")
print("QC rules       :", RULE_COUNT)
print("feature dim    :", FEATURE_DIM)

schema version : 0.1.0
modalities     : 24 -> SPR, BLI, ITC, SAXS, MST, NMR, XL-MS, cryoEM, DMS ...
QC rules       : 72
feature dim    : 64


## 1. A raw measurement in, a canonical record out

Real-world inputs are loose dicts with messy units. `standardize()` maps them to one canonical
envelope and normalizes units to SI. Here is a single-cycle SPR antibody affinity measurement.

In [2]:
raw = json.load(open("src/biophys_interop/examples/spr.json"))
raw

{'record_id': 'ex-spr-1',
 'modality': 'SPR',
 'entity_id': 'AB-001',
 'entity_kind': 'antibody',
 'entity_sequence': 'QVQLVQSGAEVKKPGASVKVSCKASGYTFT',
 'assay_type': 'SPR single-cycle kinetics',
 'doi': '10.0000/example.spr.1',
 'source_db': 'example',
 'license': 'CC0',
 'derivation': 'measured',
 'n_replicates': 3,
 'uncertainty': {'value': 3e-10, 'type': 'std', 'source': 'reported'},
 'temperature_C': 25,
 'pH': 7.4,
 'buffer': 'HBS-EP+',
 'measurement': {'KD': {'value': 2.0, 'unit': 'nM'},
  'kon': 2000000.0,
  'koff': 0.004,
  'Rmax': 80,
  'trace_conc': {'value': 5.0, 'unit': 'nM'},
  'incubation_time': {'value': 2, 'unit': 'min'}}}

In [3]:
rec = standardize(raw, "SPR")
print("entity   :", rec["entity"])
print("modality :", rec["modality"])
print("measured KD (SI, M):", rec["measurement"]["KD"])
print("conditions         :", rec["conditions"])

entity   : {'kind': 'antibody', 'id': 'AB-001', 'sequence': 'QVQLVQSGAEVKKPGASVKVSCKASGYTFT'}
modality : SPR
measured KD (SI, M): 2e-09
conditions         : {'temperature_K': 298.15, 'pH': 7.4, 'buffer': 'HBS-EP+'}


## 2. QC — biophysics judgment as code

`qc()` runs the method-aware rule registry over the record. Each fired rule carries a **severity**,
a **literature `basis`**, and an **action**. It also attaches a *calibrated* uncertainty (inflating
the reported error toward the empirically observed replicate spread).

In [4]:
rec = qc(rec)
print("flag :", rec["qc"]["flag"], "| score:", rec["qc"]["score"])
for r in rec["qc"]["reasons"]:
    print(f"  [{r['severity']:>4}] {r['code']}: {r['message']}")
    print(f"         basis: {r['basis']}")
u = rec["uncertainty"]
print("\ncalibrated uncertainty:", u["value"], f"(x{u['inflation_factor']} of reported {u['reported']})")

flag : fail | score: 0.0
  [fail] titration_regime: trace conc 5.00e-09 M >= KD 2.00e-09 M: titration regime; KD is only a lower bound.
         basis: lit-jarmoskaite-2020-measure-affinity
  [fail] equilibration: incubation 120s << 5 half-lives (866s): far from equilibrium.
         basis: lit-jarmoskaite-2020-measure-affinity
  [warn] active_fraction_unknown: active protein fraction not reported.
         basis: lit-jarmoskaite-2020-measure-affinity

calibrated uncertainty: 1.0499999999999999e-09 (x3.5 of reported 3e-10)


This SPR record **fails** QC: the trace concentration sits above KD (titration regime — KD is only a
lower bound) and the incubation is far short of equilibrium. That is exactly the kind of silent
data-quality problem that pollutes a pooled training set.

## 3. Featurize — a model-agnostic vector

`featurize()` emits a fixed-dimension numpy vector usable by any downstream model.

In [5]:
fx = featurize(rec)
v = fx["vector"]
print("dim:", fx["meta"]["dim"], "| nonzero:", int((v != 0).sum()))
print(np.round(v[:16], 3))

dim: 64 | nonzero: 10
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


## 4. Adapter — the bolt-on contract

`Adapter` defines how a frozen backbone embedding (e.g. from a protein LM) is conditioned on the
experimental feature vector. The shipped weights are **untrained reference** (`trained=False`) — the
contract validates wiring/shapes, not predictions.

In [6]:
rng = np.random.default_rng(0)
backbone_emb = rng.standard_normal(1280)      # stand-in for a frozen protein-LM embedding
ad = Adapter(backbone_dim=1280, feature_dim=FEATURE_DIM)
cond = ad(backbone_emb, v)
print("adapter:", ad)
print("trained?", ad.trained, "| trainable params:", ad.trainable_params())
print("conditioned output shape:", np.asarray(cond).shape)

adapter: Adapter(H=1280, D=64, params=83200, trained=False)
trained? False | trainable params: 83200
conditioned output shape: (1280,)


## 5. Across modalities

The same pipeline runs over every modality. Note how each example lands at a different QC verdict —
`pass`, `warn`, or `fail` — driven by its own method-aware rules.

In [7]:
import glob, os
for path in sorted(glob.glob("src/biophys_interop/examples/*.json")):
    raw = json.load(open(path))
    mod = raw.get("modality", "other")
    rec = qc(standardize(raw, mod))
    rec_for_val = {k: v for k, v in rec.items() if k != "_qc_inputs"}
    ok, errs = validate(rec_for_val)
    codes = [r["code"] for r in rec["qc"]["reasons"]]
    print(f"{os.path.basename(path):10s} {mod:6s} -> qc={rec['qc']['flag']:4s} "
          f"valid={ok}  reasons={codes}")

dls.json   DLS    -> qc=warn valid=True  reasons=['high_polydispersity', 'rh_mw_inconsistent']
dms.json   DMS    -> qc=warn valid=True  reasons=['low_read_count', 'simulated']
fida.json  FIDA   -> qc=pass valid=True  reasons=[]
saxs.json  SAXS   -> qc=fail valid=True  reasons=['guinier_upward_curvature', 'rg_inconsistent']
spr.json   SPR    -> qc=fail valid=True  reasons=['titration_regime', 'equilibration', 'active_fraction_unknown']


## Takeaway

Heterogeneous biophysical measurements become **mergeable, quality-controlled, model-ready** records
through one consistent contract — without a trained model and without exposing any private data. The
QC registry (`qc.py`, 72 cited rules) is the part that encodes domain judgment; everything else is the
plumbing that makes it composable.